In [1]:
import torch
import random
words = open('names.txt', 'r').read().splitlines()



In [2]:
random.seed(42)
random.shuffle(words)

n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

train_words = words[:n1]
dev_words   = words[n1:n2]
test_words  = words[n2:]

print("train:", len(train_words))
print("dev  :", len(dev_words))
print("test :", len(test_words))

train: 25626
dev  : 3203
test : 3204


In [4]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
V = len(stoi)

In [5]:
N2 = torch.zeros((V, V), dtype=torch.int32)

for w in train_words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        i = stoi[ch1]
        j = stoi[ch2]
        N2[i, j] += 1

P2 = (N2 + 1).float()
P2 /= P2.sum(1, keepdim=True)


In [6]:
N3 = torch.zeros((V, V, V), dtype=torch.int32)

for w in train_words:
    chs = ['.', '.'] + list(w) + ['.']
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        i = stoi[ch1]
        j = stoi[ch2]
        k = stoi[ch3]
        N3[i, j, k] += 1

P3 = (N3 + 1).float()
P3 /= P3.sum(2, keepdim=True)


In [7]:
#bigram loss-> Loss Likelihood
def eval_bigram(split_words):
    log_likelihood = 0.0
    n = 0

    for w in split_words:
        chs = ['.'] + list(w) + ['.']
        for ch1, ch2 in zip(chs, chs[1:]):
            i = stoi[ch1]
            j = stoi[ch2]
            prob = P2[i, j]
            log_likelihood += torch.log(prob)
            n += 1

    return (-log_likelihood / n).item()


In [8]:
#for trigram:
def eval_trigram(split_words):
    log_likelihood = 0.0
    n = 0

    for w in split_words:
        chs = ['.', '.'] + list(w) + ['.']
        for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
            i = stoi[ch1]
            j = stoi[ch2]
            k = stoi[ch3]
            prob = P3[i, j, k]
            log_likelihood += torch.log(prob)
            n += 1

    return (-log_likelihood / n).item()


In [9]:
print("\nBIGRAM LOSS")
print("train:", eval_bigram(train_words))
print("dev  :", eval_bigram(dev_words))
print("test :", eval_bigram(test_words))

print("\nTRIGRAM LOSS")
print("train:", eval_trigram(train_words))
print("dev  :", eval_trigram(dev_words))
print("test :", eval_trigram(test_words))


BIGRAM LOSS
train: 2.454371452331543
dev  : 2.453299045562744
test : 2.458425998687744

TRIGRAM LOSS
train: 2.2157092094421387
dev  : 2.236459732055664
test : 2.2373220920562744


In [ ]:
#Trigram performing a bit better here